# BSM L06F — Colab backbone for face enrollment

This notebook trains the shared tiny CNN backbone that the Android lab will fine-tune on-device.

Contract with Android:
- input: `96x96x3`
- backbone export: `tflite`
- embedding size: `32`
- fine-tuning target on device: `head` only


## 1. Prepare data
Use a real face dataset in Colab. The dataset must contain aligned face crops and class labels. Keep the preprocessing identical to Android: resize to `96x96`, normalize to `[0, 1]`.


In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

INPUT_SHAPE = (96, 96, 3)
EMBEDDING_SIZE = 32
NUM_CLASSES = 5
BATCH_SIZE = 32
EPOCHS = 12


In [ ]:
def build_backbone(input_shape=INPUT_SHAPE, embedding_size=EMBEDDING_SIZE):
    inputs = keras.Input(shape=input_shape, name='face_input')
    x = layers.Rescaling(1.0 / 255.0)(inputs)
    x = layers.Conv2D(16, 3, padding='same', activation='relu')(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(32, 3, padding='same', activation='relu')(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(64, 3, padding='same', activation='relu')(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(embedding_size, activation='relu', name='embedding')(x)
    return keras.Model(inputs, x, name='tiny_face_backbone')

backbone = build_backbone()
backbone.summary()


## 2. Add classification head
Train the full model in Colab first. During Android fine-tuning we will keep the backbone fixed and only adapt the head.


In [ ]:
def build_classifier(num_classes=NUM_CLASSES):
    inputs = keras.Input(shape=INPUT_SHAPE, name='face_input')
    x = layers.Rescaling(1.0 / 255.0)(inputs)
    x = layers.Conv2D(16, 3, padding='same', activation='relu')(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(32, 3, padding='same', activation='relu')(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(64, 3, padding='same', activation='relu')(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(32, activation='relu', name='embedding')(x)
    outputs = layers.Dense(num_classes, activation='softmax', name='identity_head')(x)
    return keras.Model(inputs, outputs, name='tiny_face_classifier')

model = build_classifier()
model.compile(optimizer=keras.optimizers.Adam(1e-3), loss='sparse_categorical_crossentropy', metrics=['accuracy'])


## 3. Train
Replace `train_ds` and `val_ds` with a real dataset pipeline before running training.


In [ ]:
# train_ds = ...
# val_ds = ...
# history = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS)


## 4. Export to TFLite
Export the backbone for Android. The Android app will load the exported backbone and fine-tune the head locally.


In [ ]:
# backbone_only = keras.Model(model.input, model.get_layer('embedding').output)
# converter = tf.lite.TFLiteConverter.from_keras_model(backbone_only)
# tflite_model = converter.convert()
# open('tiny_face_backbone.tflite', 'wb').write(tflite_model)


## 5. Save labels and spec
Save the class label order used in training and keep the Android input contract unchanged.


In [ ]:
# labels = ['user1', 'user2', 'user3', 'user4', 'user5']
# open('tiny_face_labels.txt', 'w').write('\n'.join(labels))
